In [1]:
from utils import *

In [2]:
def get_homogenized_modulus(**kwargs):
    """
    kwargs: {'section': string, flange/web/matrix: matrix, 'dim': []}
    - for T stringer's dimension: [0: flange width, 1: flange thickness, 2: web thickness, 3: web height]
    - for Omega stringer's dimension: [0: upper flange width, 1: web height, 2: bottom flange width, 3: thickness]
    engineering constant for axial loading is used to get homogenized modulus
    """
    # TODO: check if lateral deformation should be considered or not.
    # TODO: check the lateral deformation case for the Omega stringer.
    if kwargs['section'] == 'T':
        # two different ABD matrices for flange and web
        ABD_flange = kwargs['ABD_flange']
        ABD_web = kwargs['ABD_web']
        ABD_inv_web = np.linalg.inv(ABD_web)

        # compute the mid-line length
        flange_bm = kwargs['dim'][0]
        flange_t = kwargs['dim'][1]
        web_t = kwargs['dim'][2]
        web_bm = kwargs['dim'][3] + web_t / 2

        # flange: constrained
        E_flange_x = ABD_flange[0][0] / flange_t
        G_flange = ABD_flange[2][2] / flange_t
        nu_flange = 0

        # web: free
        E_web_x = 1 / (ABD_inv_web[0][0] * web_t)
        G_web = 1 / (ABD_inv_web[2][2] * web_t)
        nu_web = -(ABD_inv_web[0][1]) / (ABD_inv_web[0][0])

        # compute mid-line area
        flange_area = flange_bm * flange_t
        web_area = web_bm * web_t

        # compute the average modules
        E_avg = (E_flange_x * flange_area + E_web_x * web_area) / (flange_area + web_area)
        G_avg = (G_flange * flange_area + G_web * web_area) / (flange_area + web_area)
        nu_avg = (nu_flange * flange_area + nu_web * web_area) / (flange_area + web_area)

    else: # kwargs['section'] == 'Omega'
        # one uniform laminate for the omega section
        ABD_omega = kwargs['ABD_omega']
        ABD_omega_inv = np.linalg.inv(ABD_omega)

        # compute the mid-line length
        t_omega = kwargs['dim'][3]
        up_flange_bm = kwargs['dim'][0] + t_omega / 2
        web_hm = kwargs['dim'][1] - t_omega
        bot_flange_bm = kwargs['dim'][2] - t_omega

        # upper flange: constrained
        E_up_flange_x = ABD_omega[0][0] / t_omega
        G_up_flange = ABD_omega[2][2] / t_omega
        nu_up_flange = 0

        # web: free
        E_web_x = 1 / (ABD_omega_inv[0][0] * t_omega)
        G_web = 1 / (ABD_omega_inv[2][2] * t_omega)
        nu_web = -(ABD_omega_inv[0][1]) / (ABD_omega_inv[0][0])

        # bottom flange: free
        E_bot_flange_x = 1 / (ABD_omega_inv[0][0] * t_omega)
        G_bot_flange = 1 / (ABD_omega_inv[2][2] * t_omega)
        nu_bot_flange = -(ABD_omega_inv[0][1]) / (ABD_omega_inv[0][0])

        # compute mid-line area
        up_flange_area =  up_flange_bm * t_omega
        web_area = web_hm * t_omega
        bot_flange_area = bot_flange_bm * t_omega

        # compute the average elastic modules
        sum_area = 2 * up_flange_area + 2 * web_area + bot_flange_area
        E_avg = (2 * E_up_flange_x * up_flange_area + 2 * E_web_x * web_area + E_bot_flange_x * bot_flange_area) / sum_area
        G_avg = (2 * G_up_flange * up_flange_area + 2 * G_web * web_area + G_bot_flange * bot_flange_area) / sum_area
        nu_avg = (2 * nu_up_flange * up_flange_area + 2 * nu_web * web_area + nu_bot_flange * bot_flange_area) / sum_area

    return (E_avg, G_avg, nu_avg)

In [3]:
Q_mat = get_Q_matrix()

Trying to compute the homogenized elastic modules of a T-stringer

In [4]:
# TODO: DO CHECK THE SEQUENCE
stringer_seq = ['T', 'T', 'O', 'T', 'O', 'T', 'T', 'O', 'O', 'O']

flange_stack_s = [45, 45, -45, -45, 0, 0, 90, 90]
web_stack_s = [-45, -45, 45, 45, 0, 0, 90, 90]
omega_stack_s = [-45, -45, 45, 45, 0, 0, 90, 90]
t = 0.25

# compute the for T
dim_t = [70, 4, 4, 40]
ABD_flange = get_ABD_mat(Q_mat, flange_stack_s, t)
ABD_web = get_ABD_mat(Q_mat, web_stack_s, t)
hom_val_T = get_homogenized_modulus(section='T', ABD_flange=ABD_flange, ABD_web=ABD_web, dim=dim_t)
print(hom_val_T)

# compute the for
dim_omega = [10, 30, 30, 4]
ABD_omega = get_ABD_mat(Q_mat, omega_stack_s, t)
hom_val_omega = get_homogenized_modulus(section='Omega', ABD_omega=ABD_omega, dim=dim_omega)
print(hom_val_omega)

(np.float64(53637.76661254838), np.float64(19518.095536074212), np.float64(0.11106145618940035))
(np.float64(51741.94688035248), np.float64(19518.095536074212), np.float64(0.22647826360191445))
